In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

n_pedidos = 8000
np.random.seed(100)

data_pedido = [datetime.now() - timedelta(days=random.randint(1, 365)) for _ in range(n_pedidos)]
transportadoras = ['Logística Express', 'Carga Rápida S.A.', 'Transportes Nacionais', 'TransGlobal']
origens = ['Centro de Distribuição SP', 'Armazém MG', 'Fábrica SC']
destinos = ['SP', 'RJ', 'MG', 'RS', 'PR', 'SC', 'BA', 'PE', 'DF', 'GO']

df_log = pd.DataFrame({
    'id_pedido': range(50001, 50001 + n_pedidos),
    'data_pedido': data_pedido,
    'origem': np.random.choice(origens, n_pedidos, p=[0.5, 0.3, 0.2]),
    'destino_estado': np.random.choice(destinos, n_pedidos),
    'transportadora': np.random.choice(transportadoras, n_pedidos),
    'valor_produto': np.round(np.random.uniform(50, 2500, n_pedidos), 2),
    'tipo_frete': np.random.choice(['Padrão', 'Expresso'], n_pedidos, p=[0.75, 0.25])
})

# Calculando prazos estimados e reais
# Calculando prazos estimados e reais
def calcula_entregas(row):
    prazo_estimado = 3 if row['tipo_frete'] == 'Expresso' else 8
    dias_reais = prazo_estimado + np.random.choice([-2, -1, 0, 1, 2, 5, 10], p=[0.1, 0.2, 0.4, 0.15, 0.08, 0.05, 0.02])
    
    status = 'Entregue'
    if dias_reais > prazo_estimado:
        status = 'Entregue com Atraso'
    if np.random.rand() < 0.03: # 3% de chance de extravio/dano
        status = 'Extraviado/Danificado'
        
    return pd.Series([
        row['data_pedido'] + timedelta(days=prazo_estimado),
        row['data_pedido'] + timedelta(days=int(dias_reais)) if status != 'Extraviado/Danificado' else pd.NaT,
        status,
        np.round(row['valor_produto'] * (0.10 if row['tipo_frete'] == 'Padrão' else 0.20), 2) # Custo do frete
    ])

df_log[['previsao_entrega', 'data_entrega_real', 'status_entrega', 'custo_frete']] = df_log.apply(calcula_entregas, axis=1)

# Conexão segura e envio para o banco
load_dotenv()
engine = create_engine(f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASS')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}")

df_log.to_sql('supply_chain_logistics', engine, if_exists='replace', index=False)
print("Tabela 'supply_chain_logistics' criada com sucesso no PostgreSQL!")

Tabela 'supply_chain_logistics' criada com sucesso no PostgreSQL!


In [1]:
import os
from dotenv import load_dotenv

# Força o recarregamento do arquivo .env
load_dotenv(override=True)

print("--- DIAGNÓSTICO DE CONEXÃO ---")
print(f"Usuário: {os.getenv('DB_USER')}")
print(f"Host: {os.getenv('DB_HOST')}")
print(f"Porta: {os.getenv('DB_PORT')}")
print(f"Banco Alvo: {os.getenv('DB_NAME')}")
print("------------------------------")

--- DIAGNÓSTICO DE CONEXÃO ---
Usuário: postgres
Host: localhost
Porta: 5432
Banco Alvo: logistica_supply
------------------------------
